# Volume-alignment experiment (Downward Alignment) — training

**Research question:** EN (145 articles) vs PL (145 original) — does the EN–PL gap disappear when data volume is equalised?

## Three changes relative to the main training notebook

| Setting | Main notebook | This notebook |
|------|-------------|------------|
| `WARMUP_STEPS` | 1000 | **200** (data volume ~4× smaller) |
| `S3_PREFIX` | `en_augmentation_677` | **`downward_align`** |
| `make_dataframe(train)` | Load all 677 articles | **Load only sampled 145 EN articles** |

## Experimental design

- **EN**: 从 `all_train_articles_en677/` 的446 `en_*` 文章中随机采样145篇，3个seed (42, 123, 456)
- **评估**: 与论文Table 1对比 EN(446)=41.76% 和 PL(145_original) 的结果

## Expected interpretation

| Result | Interpretation |
|------|------|
| EN(145) ≈ PL(145) | Data volume is the main confound; resource paradox hypothesis weakened |
| EN(145) < PL(145) | EN still underperforms at equal volume → language composition is the main driver |
| EN(145) << EN(446) | EN benefits from more data but FP concentration is a side effect |


In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory: contains data/, results/, technique_list/, etc.
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# Training data directories
TRAIN_ARTICLES_DIR = "your/path/here"  # multilingual training articles
TRAIN_LABELS_FILE  = "your/path/here"  # corresponding label file
DEV_ARTICLES_DIR   = "your/path/here"  # dev articles
DEV_LABELS_FILE    = "your/path/here"  # dev labels

# SemEval-2023 dev data (per-language evaluation)
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirs

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


In [ ]:
import pandas as pd
from tqdm import tqdm
import os, random, tempfile, json
from os.path import isfile, join
import boto3

os.environ["TOKENIZERS_PARALLELISM"] = 'false'
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

import torch
import numpy as np
import torch.nn.functional as F

import pytorch_lightning as pl
from transformers import get_linear_schedule_with_warmup, AutoTokenizer, AutoModel
import torch.nn as nn
from torch.utils.data import DataLoader, RandomSampler
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Imports done")

## Hyperparameter configuration (3 changes annotated)

In [ ]:
model_name = 'xlm-roberta-base'
EPOCHS = 10
BATCH_SIZE = 8
ACCUMULATE_GRAD_BATCHES = 4
MAX_LENGTH = 256
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.001
LOSS_TYPE = 'bce'
MAX_EXPLANATION_LENGTH = 128

# ===== 修改1: warmup 1000 → 200 =====
WARMUP_STEPS = 200

ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET = "your-s3-bucket"
# ===== Change 2: S3 prefix =====
S3_PREFIX = "multi_label_models/downward_align"
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "your-access-key-id")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")

# ===== volume-alignment (down-aligned)专用配置 =====
EN_SAMPLE_SIZE = 145          # 与PL原始文章数对齐(SemEval subtask-3 PL native-source articles)
SEEDS = [42, 123, 456]        # 3个随机seed，报告mean±std

# 主训练数据 (all_train_articles_en677，包含所有语言)
# 文件结构:
#   en_  (446 articles)                   → EN原始, 从中采样145篇
#   po_  (~145 original + translated)→ 只保留SemEval subtask-3原始~145篇
#   ru_  (all 684 articles)               → 全部保留
#   bg_/cr_/sl_(各368篇)         → 原封不动
#   fr_/ge_/it_(43/36/62篇)      → 原封不动
# 注: fr/ge/it文章用tab分隔，其他用空格分隔
EN677_TRAIN_FOLDER = "your/path/here"
EN677_TRAIN_LABELS  = "your/path/here"

# SemEval原始PL目录: 里面的文件名是纯articleID，如 article003363.txt
# 用这个目录来精确确定哪些po_article*是subtask-3原始文章(~145篇)
# 对应的combined label ID: po_ + 文件名 = po_article003363
SEMEVAL_PL_DIR = "your/path/here"

DEV_FOLDER  = "your/path/here"
DEV_LABELS  = "your/path/here"

TECHNIQUES_FILE   = "your/path/here"
EXPLANATIONS_FILE = "your/path/here"
RESULTS_PATH      = "your/path/here"

print(f"WARMUP_STEPS : {WARMUP_STEPS}  (en677 baseline: 1000)")
print(f"EN_SAMPLE_SIZE: {EN_SAMPLE_SIZE}")
print(f"SEEDS        : {SEEDS}")
print(f"S3_PREFIX    : {S3_PREFIX}")

In [ ]:
def upload_to_s3(local_file_path, bucket_name, s3_key):
    try:
        s3 = boto3.resource('s3', endpoint_url=ENDPOINT,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY)
        print(f"Uploading to {ENDPOINT}/{bucket_name}/{s3_key}...")
        s3.Bucket(bucket_name).upload_file(local_file_path, s3_key)
        s3_uri = f"{ENDPOINT}/{bucket_name}/{s3_key}"
        print(f"Done: {s3_uri}")
        return True, s3_uri
    except Exception as e:
        print(f"Upload failed: {e}")
        return False, None

print("upload_to_s3() defined")

In [ ]:
class S3ModelCheckpoint(pl.Callback):
    """与en677完全一致，checkpoint文件名加seed以区分3次运行。"""
    def __init__(self, bucket_name, s3_prefix, seed, monitor='val_f1_micro', mode='max'):
        super().__init__()
        self.bucket_name = bucket_name
        self.s3_prefix   = s3_prefix
        self.seed        = seed
        self.monitor     = monitor
        self.mode        = mode
        self.best_model_score = None
        self.best_model_path  = None
        self.temp_dir = tempfile.mkdtemp()
        self.compare = lambda x, y: x > y if self.mode == 'max' else x < y

    def on_validation_end(self, trainer, pl_module):
        current_score = trainer.callback_metrics.get(self.monitor)
        if current_score is None: return
        if isinstance(current_score, torch.Tensor):
            current_score = current_score.item()
        if self.best_model_score is None or self.compare(current_score, self.best_model_score):
            self.best_model_score = current_score
            # seed加入文件名，避免3个run互相覆盖
            filename  = f"xlm-roberta-base_{EPOCHS}_{LOSS_TYPE}_downalign_seed{self.seed}.ckpt"
            temp_path = os.path.join(self.temp_dir, filename)
            trainer.save_checkpoint(temp_path)
            s3_key = f"{self.s3_prefix}/{filename}"
            success, s3_uri = upload_to_s3(temp_path, self.bucket_name, s3_key)
            if success:
                self.best_model_path = s3_uri
                print(f"[seed={self.seed}] Best model saved: {current_score:.4f}")
                os.remove(temp_path)

    def on_train_end(self, trainer, pl_module):
        import shutil
        try: shutil.rmtree(self.temp_dir)
        except: pass

print("S3ModelCheckpoint defined")

In [ ]:
torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using: {device} - {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
else:
    device = torch.device('cpu')
    print("Using CPU")

## Step 1: 构建训练集过滤器 `build_article_id_filter(seed)`

`all_train_articles_en677` 的文章结构：

| 前缀 | 数量 | 处理策略 |
|------|------|----------|
| `en_` | 446 articles | **随机采样145篇** (seed控制) |
| `po_` 原始 | ~145篇 | **只保留 SEMEVAL_PL_DIR 中的文章** |
| `po_` 翻译 | ~597篇 | **排除** |
| `ru_` | 740篇 | **全部保留** |
| `bg_` / `cr_` / `sl_` | 各368篇 | **原封不动** |
| `fr_` / `ge_` / `it_` | 43/36/62篇 | **排除**（误混入，非目标语言）|

注意：fr/ge/it 行在 label 文件中用 **tab** 分隔，其余用**空格**分隔。

In [ ]:
def _load_all_article_ids_from_label() -> dict:
    """
    解析 EN677_TRAIN_LABELS，返回 {prefix: set(article_ids)}。
    注意：fr/ge/it 行用 tab 分隔，其余用空格分隔。
    """
    from collections import defaultdict
    ids_by_lang = defaultdict(set)
    with open(EN677_TRAIN_LABELS) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            prefix = line.split('_')[0]
            sep = '\t' if prefix in ('fr','ge','it') else ' '
            aid = line.split(sep, 1)[0]
            ids_by_lang[prefix].add(aid)
    return ids_by_lang


def get_pl_original_ids(ids_by_lang: dict) -> set:
    """
    从 SEMEVAL_PL_DIR 读取原始PL文章ID（文件名形如 article003363.txt），
    映射到 label 文件中的 po_article003363 格式。
    只保留在 SEMEVAL_PL_DIR 中存在的 po_* 文章（即 SemEval subtask-3 原始PL文章）。
    """
    assert os.path.exists(SEMEVAL_PL_DIR), (
        f"❌ SEMEVAL_PL_DIR 不存在: {SEMEVAL_PL_DIR}")

    semeval_raw_ids = {
        f.replace('.txt','')
        for f in os.listdir(SEMEVAL_PL_DIR) if f.endswith('.txt')
    }
    print(f"  SemEval PL subtask-3 目录文章数: {len(semeval_raw_ids)}")

    # po_article003363 → raw = article003363
    pl_original = {
        aid for aid in ids_by_lang['po']
        if '_'.join(aid.split('_')[1:]) in semeval_raw_ids
    }
    print(f"  匹配到的PL原始文章数: {len(pl_original)}")
    if len(pl_original) == 0:
        raise RuntimeError(
            "❌ 未找到任何PL原始文章！请检查:\n"
            f"   SEMEVAL_PL_DIR: {SEMEVAL_PL_DIR}\n"
            f"   示例SemEval ID: {sorted(semeval_raw_ids)[:3]}\n"
            f"   label文件中po_示例: {sorted(ids_by_lang['po'])[:3]}"
        )
    return pl_original


def build_article_id_filter(seed: int) -> dict:
    """
    构建volume-alignment (down-aligned)实验的 article_id_filter (set)。

    保留: EN(145采样) | PL(subtask-3原始~145) | RU(全量) | BG/CR/SL(原封不动)
    排除: PO翻译文章 | FR/GE/IT（误混入，不属于目标语言）
    """
    print(f"\n{'='*55}")
    print(f"build_article_id_filter | seed={seed}")
    print(f"{'='*55}")

    ids_by_lang = _load_all_article_ids_from_label()

    # 1. EN: 随机采样145篇
    en_all = sorted(ids_by_lang['en'])
    print(f"  EN原始总数: {len(en_all)} 篇  → 采样 {EN_SAMPLE_SIZE} 篇")
    assert len(en_all) >= EN_SAMPLE_SIZE
    random.seed(seed)
    en_sampled = set(random.sample(en_all, EN_SAMPLE_SIZE))

    # 2. PL: 只保留 SemEval subtask-3 原始文章（排除翻译）
    print(f"  PL过滤 (SEMEVAL_PL_DIR → po_article*):")
    pl_original = get_pl_original_ids(ids_by_lang)

    # 3. RU: 全量保留
    ru_all = set(ids_by_lang['ru'])
    print(f"  RU全量: {len(ru_all)} 篇")

    # 4. BG / CR / SL: 原封不动
    other_langs = set()
    for lang in ('bg', 'cr', 'sl'):
        other_langs |= ids_by_lang[lang]
    print(f"  其他语言 (bg/cr/sl): {len(other_langs)} 篇")

    # 5. FR / GE / IT: 排除（误混入，不属于目标语言）
    fr_ge_it_count = sum(len(ids_by_lang[l]) for l in ('fr','ge','it'))
    print(f"  FR/GE/IT: {fr_ge_it_count} 篇 → 排除")

    combined = en_sampled | pl_original | ru_all | other_langs
    print(f"\n  总计: EN={len(en_sampled)} + PL={len(pl_original)}"
          f" + RU={len(ru_all)} + bg/cr/sl={len(other_langs)} = {len(combined)} 篇")

    return {
        'filter'   : combined,
        'en_ids'   : en_sampled,
        'pl_ids'   : pl_original,
        'ru_ids'   : ru_all,
        'other_ids': other_langs,
    }


print("build_article_id_filter defined")

## Step 2: make_dataframe — 修改3

训练时接收 `article_id_filter`，只加载采样的145篇；dev路径不变。

In [ ]:
def make_dataframe(data_type='train', article_id_filter=None):
    """
    与 en677_train.ipynb 完全一致，增加可选的 article_id_filter 参数。
    注意: EN677_TRAIN_LABELS 中 fr/ge/it 行用 tab 分隔，其余用空格分隔。
    article_id_filter: set of article IDs to keep (训练时使用，dev不过滤)。
    """
    if data_type == 'train':
        input_folder = EN677_TRAIN_FOLDER
        labels_fn    = EN677_TRAIN_LABELS
    elif data_type == 'dev':
        input_folder = DEV_FOLDER
        labels_fn    = DEV_LABELS
    else:
        raise ValueError("data_type must be 'train' or 'dev'")

    print(f"Loading {data_type}: {input_folder}")
    if article_id_filter:
        print(f"  Filter: {len(article_id_filter)} article IDs")

    text    = []
    skipped = 0
    for fil in tqdm(filter(lambda x: x.endswith('.txt'), os.listdir(input_folder))):
        iD = fil.split('.')[0]
        # ===== 修改3: 训练时只读过滤后的文章 =====
        if data_type == 'train' and article_id_filter and iD not in article_id_filter:
            continue
        try:
            with open(input_folder + fil, 'r', encoding='utf-8') as f:
                content = f.read()
            lines = list(enumerate(content.splitlines(), 1))
            text.extend([(iD,) + line for line in lines])
        except UnicodeDecodeError:
            skipped += 1
    if skipped:
        print(f"Skipped {skipped} files (UnicodeDecodeError)")

    df_text = pd.DataFrame(text, columns=['id', 'line', 'text'])
    df_text.line = df_text.line.apply(int)
    df_text = df_text[df_text.text.str.strip().str.len() > 0].copy()
    df_text = df_text.set_index(['id', 'line'])

    try:
        labels_data = []
        with open(labels_fn, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                # 混合分隔符：fr/ge/it用tab，其余用空格
                prefix = line.split('_')[0]
                sep = '\t' if prefix in ('fr','ge','it') else ' '
                parts = line.split(sep, 2)
                if len(parts) < 2: continue
                try:
                    aid  = parts[0]
                    lnum = int(parts[1])
                    lbls = parts[2].strip() if len(parts) >= 3 else ''
                    if data_type == 'train' and article_id_filter and aid not in article_id_filter:
                        continue
                    labels_data.append([aid, lnum, lbls])
                except ValueError:
                    pass
        if labels_data:
            ldf = pd.DataFrame(labels_data, columns=['id', 'line', 'labels'])
            ldf['line'] = ldf['line'].astype(int)
            ldf = ldf.set_index(['id', 'line'])
            df  = df_text.join(ldf, how='left')[['text', 'labels']]
            df['labels'] = df['labels'].fillna('')
        else:
            raise ValueError("Empty labels after filter")
    except Exception as e:
        print(f"Label error: {e}")
        df = df_text.copy()
        df['labels'] = ''
        df = df[['text', 'labels']]

    df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
    return df.reset_index()

print("make_dataframe() defined (mixed delimiter + article_id_filter)")

## 以下Cell与 en677_train.ipynb 完全一致

In [ ]:
TECHNIQUES_FILE = "your/path/here"

def load_label_classes():
    with open(TECHNIQUES_FILE, "r") as f:
        labels_name = [line.rstrip() for line in f if line.rstrip()]
    labels_name.sort()
    print(f"Loaded {len(labels_name)} techniques from {TECHNIQUES_FILE}")
    return labels_name

def load_explanations_data(explanations_file):
    explanations = {}
    try:
        df = pd.read_csv(explanations_file, sep='\t')
        for _, row in df.iterrows():
            explanations[(row['id'], row['text'])] = row['analysis']
    except Exception as e:
        print(f"Error: {e}")
    print(f"Loaded {len(explanations)} explanations")
    return explanations

labels = load_label_classes()
print(f"Labels: {labels[:3]}... total={len(labels)}")

In [ ]:
def load_multi_label_data_with_explanations(data_type='train', explanations=None,
                                              article_id_filter=None):
    df = make_dataframe(data_type=data_type, article_id_filter=article_id_filter)
    df = df[df['text'].str.len() <= 1000].copy()

    all_idxs = df["id"].to_numpy()
    all_lines = df["line"].to_numpy()
    all_data  = [str(t) for t in df["text"].to_numpy()]

    labels_name = load_label_classes()
    num_labels  = len(labels_name)

    multi_labels = []
    for label_str in df['labels'].fillna('').values:
        label_vec = np.zeros(num_labels, dtype=np.float32)
        if label_str:
            for label in label_str.split(','):
                if label in labels_name:
                    label_vec[labels_name.index(label)] = 1.0
        multi_labels.append(label_vec)

    if explanations:
        explanation_texts = [explanations.get((idx, text), "")
                             for idx, text in zip(all_idxs, all_data)]
    else:
        explanation_texts = [""] * len(all_data)
    explanation_texts = [str(e) if e else "" for e in explanation_texts]

    return (np.array(all_idxs, dtype=object), np.array(all_lines, dtype=np.int32),
            all_data, torch.tensor(np.array(multi_labels)), explanation_texts, labels_name)

print("load_multi_label_data_with_explanations() defined")

In [ ]:
class MultiLabelClassificationDataWithExplanations(torch.utils.data.Dataset):
    def __init__(self, tokenizer=None, max_length=MAX_LENGTH,
                 max_explanation_length=MAX_EXPLANATION_LENGTH,
                 data_tuple=None, data_type='train'):
        self.max_length = max_length
        self.max_explanation_length = max_explanation_length
        if data_tuple is None:
            self.idxs, self.lines, self.texts, self.y, self.explanations, self.label_names =                 load_multi_label_data_with_explanations(data_type)
        else:
            self.idxs, self.lines, self.texts, self.y, self.explanations, self.label_names = data_tuple
        self.tokenized = False
        if tokenizer is not None:
            self.tokenized = True
            bs = 256
            self.texts = [str(t) for t in self.texts]
            self.text_input_ids, self.text_attention_mask = [], []
            for i in range(0, len(self.texts), bs):
                tok = tokenizer(self.texts[i:i+bs], padding="max_length", truncation=True,
                                max_length=self.max_length, return_tensors="pt")
                self.text_input_ids.append(tok["input_ids"])
                self.text_attention_mask.append(tok["attention_mask"])
            self.text_input_ids     = torch.cat(self.text_input_ids, dim=0)
            self.text_attention_mask = torch.cat(self.text_attention_mask, dim=0)
            self.exp_input_ids, self.exp_attention_mask = [], []
            exps = [e if e and e.strip() else "no explanation" for e in self.explanations]
            for i in range(0, len(exps), bs):
                tok = tokenizer(exps[i:i+bs], padding="max_length", truncation=True,
                                max_length=self.max_explanation_length, return_tensors="pt")
                self.exp_input_ids.append(tok["input_ids"])
                self.exp_attention_mask.append(tok["attention_mask"])
            self.exp_input_ids      = torch.cat(self.exp_input_ids, dim=0)
            self.exp_attention_mask = torch.cat(self.exp_attention_mask, dim=0)
            if hasattr(torch.cuda, 'empty_cache'): torch.cuda.empty_cache()

    def __getitem__(self, index):
        return (self.text_input_ids[index], self.text_attention_mask[index],
                self.exp_input_ids[index],  self.exp_attention_mask[index],
                torch.squeeze(self.y[index]), self.idxs[index],
                torch.tensor(self.lines[index], dtype=torch.int64))

    def __len__(self):
        return self.text_input_ids.shape[0] if self.tokenized else len(self.texts)

print("Dataset class defined")

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, pos_weight=None, reduction='mean'):
        super().__init__()
        self.alpha, self.gamma, self.pos_weight, self.reduction = alpha, gamma, pos_weight, reduction
    def forward(self, inputs, targets):
        eps = 1e-7
        targets_s = torch.clamp(targets.float(), min=eps, max=1-eps)
        bce = (F.binary_cross_entropy_with_logits(inputs, targets_s,
               pos_weight=self.pos_weight, reduction='none')
               if self.pos_weight is not None
               else F.binary_cross_entropy_with_logits(inputs, targets_s, reduction='none'))
        pt = torch.exp(-bce)
        fl = self.alpha * (1-pt)**self.gamma * bce
        return torch.mean(fl) if self.reduction=='mean' else (
               torch.sum(fl)  if self.reduction=='sum'  else fl)

def compute_multi_label_class_weights(labels, neg_pos_ratio=3.0, max_weight=30.0):
    pos_counts = labels.sum(dim=0)
    total      = labels.shape[0]
    neg_counts = total - pos_counts
    weights = []
    for i in range(labels.shape[1]):
        weights.append(min((neg_counts[i]/pos_counts[i])*neg_pos_ratio, max_weight)
                       if pos_counts[i] > 0 else 1.0)
    return torch.tensor(weights, dtype=torch.float32, device=labels.device)

print("FocalLoss + compute_multi_label_class_weights defined")

In [ ]:
class MultiLabelClassifierWithExplanations(pl.LightningModule):
    # ===== 与 en677_train.ipynb 完全一致 =====
    def __init__(self, plm, num_labels, class_weights=None, learning_rate=LEARNING_RATE,
                 warmup_steps=WARMUP_STEPS, loss_type='bce', focal_gamma=2.0, focal_alpha=1.0):
        super().__init__()
        self.text_encoder        = AutoModel.from_pretrained(plm)
        self.explanation_encoder = AutoModel.from_pretrained(plm)
        if hasattr(self.text_encoder, "gradient_checkpointing_enable"):
            self.text_encoder.gradient_checkpointing_enable()
            self.explanation_encoder.gradient_checkpointing_enable()
        self.hidden_size  = self.text_encoder.config.hidden_size
        self.num_labels   = num_labels
        self.learning_rate = learning_rate
        self.warmup_steps  = warmup_steps
        self.loss_type     = loss_type
        self.text_to_exp_attention = nn.MultiheadAttention(
            embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        self.exp_to_text_attention = nn.MultiheadAttention(
            embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        self.fusion_layer = nn.Sequential(
            nn.Linear(self.hidden_size*4, self.hidden_size*2),
            nn.LayerNorm(self.hidden_size*2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(self.hidden_size*2, self.hidden_size),
            nn.LayerNorm(self.hidden_size),  nn.ReLU(), nn.Dropout(0.1))
        self.classifier = nn.Linear(self.hidden_size, num_labels)
        if loss_type == 'focal':
            self.criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma,
                                       pos_weight=class_weights)
        else:
            self.criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
        self.train_preds, self.train_labels = [], []
        self.val_preds,   self.val_labels   = [], []

    def forward(self, text_ids, text_mask, exp_ids=None, exp_mask=None):
        text_out = self.text_encoder(input_ids=text_ids, attention_mask=text_mask,
                                     return_dict=True)
        text_h   = text_out.last_hidden_state
        text_cls = text_h[:, 0, :]
        if exp_ids is None or exp_mask is None:
            return self.classifier(text_cls)
        exp_out = self.explanation_encoder(input_ids=exp_ids, attention_mask=exp_mask,
                                           return_dict=True)
        exp_h   = exp_out.last_hidden_state
        exp_cls = exp_h[:, 0, :]
        t2e, _ = self.text_to_exp_attention(
            query=text_h, key=exp_h, value=exp_h,
            key_padding_mask=~exp_mask.bool())
        e2t, _ = self.exp_to_text_attention(
            query=exp_h, key=text_h, value=text_h,
            key_padding_mask=~text_mask.bool())
        combined = torch.cat([text_cls, exp_cls, t2e[:, 0, :], e2t[:, 0, :]], dim=1)
        return self.classifier(self.fusion_layer(combined))

    def _compute_loss(self, preds, labels):
        if labels.shape != preds.shape:
            if len(labels.shape) < len(preds.shape): labels = labels.view(*preds.shape)
            else: preds = preds.view(*labels.shape)
        return self.criterion(torch.clamp(preds.float(), min=1e-7, max=1-1e-7),
                              torch.clamp(labels.float(), min=1e-7, max=1-1e-7))

    def training_step(self, batch, batch_idx):
        text_ids, text_mask, exp_ids, exp_mask, labels, _, _ = batch
        preds = self(text_ids=text_ids, text_mask=text_mask, exp_ids=exp_ids, exp_mask=exp_mask)
        loss  = self._compute_loss(preds, labels)
        if torch.isnan(loss):
            return {"loss": torch.tensor(1e-5, requires_grad=True, device=loss.device)}
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        with torch.no_grad():
            pp = torch.sigmoid(preds)
            if not torch.isnan(pp).any():
                self.train_preds.extend(pp.detach().cpu().numpy())
                self.train_labels.extend(labels.detach().cpu().numpy())
        if batch_idx % 50 == 0 and hasattr(torch.cuda, 'empty_cache'):
            torch.cuda.empty_cache()
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        text_ids, text_mask, exp_ids, exp_mask, labels, _, _ = batch
        preds = self(text_ids=text_ids, text_mask=text_mask, exp_ids=exp_ids, exp_mask=exp_mask)
        loss  = self._compute_loss(preds, labels)
        if torch.isnan(loss):
            return {"val_loss": torch.tensor(0.0, device=self.device)}
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        with torch.no_grad():
            pp  = torch.sigmoid(preds)
            pl_ = (pp > 0.5).float()
            self.log('val_exact_match', (pl_ == labels).all(dim=1).float().mean(), prog_bar=True)
            if not torch.isnan(pp).any():
                self.val_preds.extend(pp.detach().cpu().numpy())
                self.val_labels.extend(labels.detach().cpu().numpy())
        return {"val_loss": loss}

    def _log_metrics(self, preds_list, labels_list, prefix):
        from sklearn.metrics import f1_score, precision_score, recall_score
        pred_l = (np.array(preds_list) > 0.5).astype(int)
        true_l = np.array(labels_list)
        for avg in ['micro', 'macro']:
            is_prog = (prefix == 'val' and avg == 'micro')
            self.log(f'{prefix}_f1_{avg}',
                     f1_score(true_l, pred_l, average=avg, zero_division=0), prog_bar=is_prog)
            self.log(f'{prefix}_precision_{avg}',
                     precision_score(true_l, pred_l, average=avg, zero_division=0))
            self.log(f'{prefix}_recall_{avg}',
                     recall_score(true_l, pred_l, average=avg, zero_division=0))
        if prefix == 'val' and preds_list:
            for i in range(self.num_labels):
                lf1 = f1_score(true_l[:, i], pred_l[:, i], zero_division=0)
                if lf1 > 0: self.log(f'val_f1_label_{i}', lf1)

    def on_train_epoch_end(self):
        self._log_metrics(self.train_preds, self.train_labels, 'train')
        self.train_preds, self.train_labels = [], []

    def on_validation_epoch_end(self):
        self._log_metrics(self.val_preds, self.val_labels, 'val')
        self.val_preds, self.val_labels = [], []

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.learning_rate,
                                weight_decay=WEIGHT_DECAY)
        sch = get_linear_schedule_with_warmup(
            opt, num_warmup_steps=self.warmup_steps,
            num_training_steps=self.trainer.estimated_stepping_batches)
        return {"optimizer": opt,
                "lr_scheduler": {"scheduler": sch, "interval": "step",
                                 "frequency": 1, "monitor": "val_f1_micro"}}

print("Model defined  (warmup_steps defaults to WARMUP_STEPS=200)")

## Step 3: 单Seed训练函数

In [ ]:
def train_downalign_model(seed: int, explanations_file=None, loss_type=LOSS_TYPE):
    """
    volume-alignment (down-aligned)实验的单seed训练。
    与 en677_train 的 train_en677_model() 完全对应，
    区别：传入 article_id_filter (EN145+PL原始+RU全量) 给数据加载函数。
    """
    print("=" * 60)
    print(f"Downward Alignment Training | seed={seed} | warmup={WARMUP_STEPS}")
    print(f"  EN={EN_SAMPLE_SIZE}采样 | PL=original-only | RU=full")
    print("=" * 60)

    # ===== 核心: 构建三语言过滤器 =====
    id_filter_info = build_article_id_filter(seed=seed)
    article_id_filter = id_filter_info['filter']

    labels_name = load_label_classes()
    num_labels  = len(labels_name)
    print(f"Labels: {num_labels}, Loss: {loss_type}")

    explanations = None
    if explanations_file and os.path.exists(explanations_file):
        explanations = load_explanations_data(explanations_file)

    tokenizer    = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    temp_log_dir = tempfile.mkdtemp(prefix=f"lightning_logs_seed{seed}_")

    print(f"Loading train data (EN={len(id_filter_info['en_ids'])} "
          f"PL={len(id_filter_info['pl_ids'])} "
          f"RU={len(id_filter_info['ru_ids'])} 篇)...")
    idxs, lines, X, y, exp_train, _ = load_multi_label_data_with_explanations(
        'train', explanations, article_id_filter=article_id_filter)
    print(f"Train samples (paragraphs): {len(X)}")

    print("Loading dev data...")
    d_idxs, d_lines, d_X, d_y, exp_dev, _ = load_multi_label_data_with_explanations(
        'dev', explanations)
    print(f"Dev samples: {len(d_X)}")

    class_weights = compute_multi_label_class_weights(y).to(device)

    ds_train = MultiLabelClassificationDataWithExplanations(
        tokenizer=tokenizer, data_tuple=(idxs, lines, X, y, exp_train, labels_name))
    ds_val   = MultiLabelClassificationDataWithExplanations(
        tokenizer=tokenizer, data_tuple=(d_idxs, d_lines, d_X, d_y, exp_dev, labels_name))

    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE,
                              sampler=RandomSampler(ds_train), num_workers=2, pin_memory=True)
    val_loader   = DataLoader(ds_val,   batch_size=BATCH_SIZE*2, num_workers=2, pin_memory=True)

    model = MultiLabelClassifierWithExplanations(
        plm=model_name, num_labels=num_labels, class_weights=class_weights,
        learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, loss_type=loss_type)

    s3_cb = S3ModelCheckpoint(bucket_name=S3_BUCKET, s3_prefix=S3_PREFIX,
                              seed=seed, monitor='val_f1_micro', mode='max')
    es_cb = EarlyStopping(monitor='val_f1_micro', patience=3, verbose=True, mode='max')

    try:
        import tensorboard
        logger = TensorBoardLogger(save_dir=temp_log_dir, name=f'downalign_seed{seed}')
    except ImportError:
        from pytorch_lightning.loggers import CSVLogger
        logger = CSVLogger(save_dir=temp_log_dir, name=f'downalign_seed{seed}')

    trainer = pl.Trainer(
        max_epochs=EPOCHS, callbacks=[s3_cb, es_cb], precision="32",
        accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES, gradient_clip_val=0.5,
        accelerator="auto", devices=1, enable_progress_bar=True,
        enable_model_summary=(seed == SEEDS[0]),
        log_every_n_steps=10, logger=logger)

    result = {
        "seed": seed,
        "best_val_f1_micro": None,
        "best_model_path": None,
        "n_train_paragraphs": len(X),
        "en_articles": len(id_filter_info['en_ids']),
        "pl_articles": len(id_filter_info['pl_ids']),
        "ru_articles": len(id_filter_info['ru_ids']),
    }
    try:
        model = model.to(device)
        trainer.fit(model, train_loader, val_loader)
        result["best_val_f1_micro"] = float(s3_cb.best_model_score) if s3_cb.best_model_score else None
        result["best_model_path"]   = s3_cb.best_model_path
        print(f"\n✅ seed={seed} | Best val_f1_micro: {result['best_val_f1_micro']:.4f}")
        print(f"   EN={result['en_articles']} | PL={result['pl_articles']} | RU={result['ru_articles']}")
    except Exception as e:
        print(f"Training failed (seed={seed}): {e}")
        import traceback; traceback.print_exc()
    finally:
        if hasattr(torch.cuda, 'empty_cache'): torch.cuda.empty_cache()
        try: del model, train_loader, val_loader, ds_train, ds_val
        except: pass
        import gc; gc.collect()

    try:
        import shutil; shutil.rmtree(temp_log_dir)
    except: pass

    return result

print("train_downalign_model() defined")

## Sanity Check — 训练前运行

In [ ]:
print("=" * 60)
print("Sanity Check")
print("=" * 60)

# 1. 路径检查
for name, path in [("EN677_TRAIN_FOLDER", EN677_TRAIN_FOLDER),
                   ("EN677_TRAIN_LABELS",  EN677_TRAIN_LABELS),
                   ("SEMEVAL_PL_DIR",      SEMEVAL_PL_DIR),
                   ("DEV_FOLDER",          DEV_FOLDER),
                   ("DEV_LABELS",          DEV_LABELS),
                   ("TECHNIQUES_FILE",     TECHNIQUES_FILE),
                   ("EXPLANATIONS_FILE",   EXPLANATIONS_FILE)]:
    ok = os.path.exists(path)
    n  = f"({len(os.listdir(path))} files)" if ok and os.path.isdir(path) else ""
    print(f"  {'✅' if ok else '❌'} {name}: {path} {n}")

# 2. label文件前缀分布
print("\nlabel文件前缀分布:")
ids = _load_all_article_ids_from_label()
for lang in sorted(ids.keys()):
    excluded = " ← 排除" if lang in ('fr','ge','it') else ""
    print(f"  {lang}: {len(ids[lang])} 篇{excluded}")

# 3. 完整过滤器验证
print("\n测试 build_article_id_filter(seed=42)...")
filt = build_article_id_filter(seed=42)
print(f"\n结果:")
print(f"  EN (采样{EN_SAMPLE_SIZE}): {len(filt['en_ids'])} 篇  {'✅' if len(filt['en_ids'])==EN_SAMPLE_SIZE else '❌'}")
print(f"  PL (原始):     {len(filt['pl_ids'])} 篇")
print(f"  RU (全量):     {len(filt['ru_ids'])} 篇")
print(f"  bg/cr/sl:      {len(filt['other_ids'])} 篇")
print(f"  总article数:   {len(filt['filter'])} 篇")

# 4. 验证FR/GE/IT被排除
fr_ge_it_in_filter = {a for a in filt['filter']
                       if a.startswith('fr_') or a.startswith('ge_') or a.startswith('it_')}
print(f"\n  FR/GE/IT混入filter: {len(fr_ge_it_in_filter)} 篇  "
      f"{'✅ 0篇，排除正确' if len(fr_ge_it_in_filter)==0 else '❌ 未排除！'}")

# 5. 验证PO翻译文章被排除
source_raw = set()
for lang in ('fr','ge','it'):
    for aid in ids[lang]:
        source_raw.add('_'.join(aid.split('_')[1:]))
po_translated_in_filter = {a for a in filt['pl_ids']
                            if '_'.join(a.split('_')[1:]) in source_raw}
print(f"  PO翻译文章混入filter: {len(po_translated_in_filter)} 篇  "
      f"{'✅ 0篇，过滤正确' if len(po_translated_in_filter)==0 else '❌ 存在翻译文章！'}")

# 6. seed可重复性
f42a = build_article_id_filter(42)['en_ids']
f42b = build_article_id_filter(42)['en_ids']
f123 = build_article_id_filter(123)['en_ids']
assert f42a == f42b, "❌ 同seed结果不一致"
assert f42a != f123, "❌ 不同seed结果相同"
print(f"  seed可重复性: ✅")
print(f"  seed42/123 EN重叠: {len(f42a & f123)} 篇")
print("\n✅ Sanity check 通过")

## 运行全部 3 个 Seeds

In [ ]:
all_results = []

for seed in SEEDS:
    print(f"\n{'#'*70}")
    print(f"# Seed {seed} / {SEEDS}")
    print(f"{'#'*70}")
    result = train_downalign_model(
        seed=seed,
        explanations_file=EXPLANATIONS_FILE,
        loss_type=LOSS_TYPE)
    all_results.append(result)
    print(f"[seed={seed}] val_f1_micro = {result['best_val_f1_micro']}")

# 持久化
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f"\n结果已保存: {RESULTS_PATH}")

## 结果汇总与解读

In [ ]:
print("=" * 70)
print("volume-alignment (down-aligned)实验结果汇总")
print("=" * 70)
print(f"配置: EN={EN_SAMPLE_SIZE}篇(采样) | warmup={WARMUP_STEPS} | seeds={SEEDS}")
print()

scores = [r["best_val_f1_micro"] for r in all_results if r["best_val_f1_micro"] is not None]

if scores:
    mean_f1 = np.mean(scores)
    std_f1  = np.std(scores)
    print("┌─────────────────────────────────────────────────────────┐")
    for r in all_results:
        print(f"│  seed={r['seed']}  val_f1_micro={r['best_val_f1_micro']:.4f}  "
              f"EN={r['en_articles']} PL={r['pl_articles']} RU={r['ru_articles']}  │")
    print(f"│  Mean ± Std : {mean_f1:.4f} ± {std_f1:.4f}                        │")
    print("└─────────────────────────────────────────────────────────┘")
    print()
    print("对比基线 (论文Table 1, Sup-FT, val_f1_micro):")
    print("  EN(446) full  [paper]:   59.15%")
    print("  PL(674) full:   66.75%")
    print(f"  EN(145) down: {mean_f1*100:.2f}% ± {std_f1*100:.2f}%")
else:
    print("⚠ 无有效结果")

## 后续: 按语言Macro F1评估

训练结束后用以下步骤得到论文需要的 **EN(145) Macro F1** 对比数字：

1. 从S3下载最优checkpoint（3个seed各一个）
2. 加载模型，在 `all_dev_articles/` 中按前缀分组：
   - `en_*` → EN dev set (90篇)
   - `po_*` → PL dev set (49篇)
3. 分别计算 Macro F1，报告 mean±std across 3 seeds

对比目标（来自 `各个方法checkthat2024的结果`）：
- EN(446) Sup-FT Macro F1 = **41.76%**
- PL(145 original) Sup-FT Macro F1 = **(待实验)**


## 按语言评估 — EN & PL Macro/Micro F1

使用段落级金标准（与论文Table 1相同的评估协议）：
- EN gold: `en_dev-labels-subtask-3.txt` (90篇, 格式: `article_id \t line \t techniques`)
- PL gold: `po_dev-labels-subtask-3.txt` (49篇)

**article_id映射规则：**
- Gold EN: `813452859` → dev folder: `en_article813452859.txt`
- Gold PO: `25106` → dev folder: `po_article25106.txt`


In [ ]:
import boto3, tempfile, os
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from collections import defaultdict
from tqdm import tqdm

# ===== 路径配置 =====
EN_GOLD = "your/path/here"
PO_GOLD = "your/path/here"
CKPT_S3_KEY = "multi_label_models/downward_align/xlm-roberta-base_10_bce_downalign_seed42.ckpt"
CKPT_LOCAL  = "your/path/here"

os.makedirs("your/path/here", exist_ok=True)

# 先把gold文件复制过去（如果在服务器上运行，需要手动上传或从此处写入）
# 这里直接硬编码gold文件路径（假设已上传）
print("路径配置完成")
print(f"  EN_GOLD : {EN_GOLD}")
print(f"  PO_GOLD : {PO_GOLD}")
print(f"  CKPT    : {CKPT_LOCAL}")

In [ ]:
# ===== Step 1: 从S3下载checkpoint =====
def download_ckpt_from_s3(s3_key, local_path):
    if os.path.exists(local_path):
        print(f"✅ checkpoint已存在: {local_path}")
        return True
    print(f"从S3下载: {s3_key}")
    try:
        s3 = boto3.resource('s3', endpoint_url=ENDPOINT,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY)
        s3.Bucket(S3_BUCKET).download_file(s3_key, local_path)
        print(f"✅ 下载完成: {local_path}")
        return True
    except Exception as e:
        print(f"❌ 下载失败: {e}")
        return False

download_ckpt_from_s3(CKPT_S3_KEY, CKPT_LOCAL)

In [ ]:
# ===== Step 2: 加载gold label文件 =====
# 格式: article_id \t line_num \t technique1,technique2,...
# 返回: dict { (article_id_raw, line_num) -> set(techniques) }

def load_gold_labels(gold_path):
    """
    返回:
      para_labels: dict { (raw_article_id, line_num): set(techniques) }
      articles: set of raw_article_ids
    """
    para_labels = {}
    articles = set()
    with open(gold_path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split('\t')
            raw_id  = parts[0]                               # 纯数字, 如 '813452859'
            lnum    = int(parts[1])
            techs   = set(parts[2].split(',')) - {''} if len(parts) >= 3 else set()
            para_labels[(raw_id, lnum)] = techs
            articles.add(raw_id)
    print(f"Gold: {len(articles)} 篇文章, {len(para_labels)} 段落")
    return para_labels, articles

print("Loading EN gold...")
en_gold, en_articles = load_gold_labels(EN_GOLD)
print("Loading PO gold...")
po_gold, po_articles = load_gold_labels(PO_GOLD)

In [ ]:
# ===== Step 3: 加载模型并推理 =====
# 重用训练时定义的模型类（前面已经定义好了）

def load_model_from_ckpt(ckpt_path, num_labels, device):
    model = MultiLabelClassifierWithExplanations.load_from_checkpoint(
        ckpt_path,
        plm=model_name,
        num_labels=num_labels,
        class_weights=None,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        loss_type=LOSS_TYPE,
        map_location=device
    )
    model.eval()
    model.to(device)
    print(f"✅ 模型加载完成 (device={device})")
    return model


def predict_language(model, tokenizer, labels_name, dev_folder, dev_labels_path,
                     lang_prefix, lang_raw_ids, gold_labels):
    """
    对指定语言的dev文章进行段落级推理，返回 (y_true, y_pred) 两个矩阵。

    lang_prefix: 'en_' 或 'po_'
    lang_raw_ids: gold文件里的纯数字article_id集合
    gold_labels: { (raw_id, line_num) -> set(techniques) }
    """
    # 构建 dev_article_id → raw_id 的映射
    # dev文件名: en_article813452859.txt → raw_id = '813452859'
    # po文件名:  po_article25106.txt    → raw_id = '25106'
    prefix_full = lang_prefix                        # 'en_' 或 'po_'
    middle      = 'article'

    # 读取dev folder所有该语言的文章
    all_dev_files = [f for f in os.listdir(dev_folder)
                     if f.endswith('.txt') and f.startswith(prefix_full)]
    # 只处理gold中出现的文章
    target_files = []
    for fname in all_dev_files:
        # en_article813452859.txt → raw_id = 813452859
        raw_id = fname.replace(prefix_full + middle, '').replace('.txt', '')
        if raw_id in lang_raw_ids:
            target_files.append((fname, raw_id))

    print(f"  {lang_prefix}: {len(target_files)} 篇文章需要推理")

    num_labels = len(labels_name)
    label2idx  = {l: i for i, l in enumerate(labels_name)}

    all_true, all_pred = [], []

    for fname, raw_id in tqdm(target_files, desc=f"推理 {lang_prefix}"):
        fpath = os.path.join(dev_folder, fname)
        with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        para_list = list(enumerate(content.splitlines(), 1))
        para_list = [(lnum, text) for lnum, text in para_list
                     if text.strip() and len(text.strip()) <= 1000]

        if not para_list: continue

        texts = [text for _, text in para_list]
        lnums = [lnum for lnum, _ in para_list]

        # tokenize
        tok = tokenizer(texts, padding='max_length', truncation=True,
                        max_length=MAX_LENGTH, return_tensors='pt')
        # explanation: 空字符串（推理时不用explanation也可，用"no explanation"）
        exp_texts = ['no explanation'] * len(texts)
        exp_tok = tokenizer(exp_texts, padding='max_length', truncation=True,
                            max_length=MAX_EXPLANATION_LENGTH, return_tensors='pt')

        text_ids   = tok['input_ids'].to(device)
        text_mask  = tok['attention_mask'].to(device)
        exp_ids    = exp_tok['input_ids'].to(device)
        exp_mask   = exp_tok['attention_mask'].to(device)

        with torch.no_grad():
            logits = model(text_ids=text_ids, text_mask=text_mask,
                           exp_ids=exp_ids, exp_mask=exp_mask)
            probs  = torch.sigmoid(logits).cpu().numpy()

        # 对每个段落，用0.5阈值得到预测标签
        preds = (probs > 0.5).astype(int)

        for i, lnum in enumerate(lnums):
            # gold标签向量
            gold_techs = gold_labels.get((raw_id, lnum), set())
            true_vec = np.zeros(num_labels, dtype=int)
            for t in gold_techs:
                if t in label2idx:
                    true_vec[label2idx[t]] = 1

            all_true.append(true_vec)
            all_pred.append(preds[i])

    return np.array(all_true), np.array(all_pred)


print("推理函数定义完成")

In [ ]:
# ===== Step 4: 执行评估 =====
labels_name = load_label_classes()
num_labels  = len(labels_name)
tokenizer   = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# 加载模型
eval_model = load_model_from_ckpt(CKPT_LOCAL, num_labels, device)

# EN推理
print("\n推理 EN dev set...")
en_true, en_pred = predict_language(
    eval_model, tokenizer, labels_name,
    DEV_FOLDER, DEV_LABELS,
    lang_prefix='en_', lang_raw_ids=en_articles, gold_labels=en_gold)

# PO推理
print("\n推理 PO dev set...")
po_true, po_pred = predict_language(
    eval_model, tokenizer, labels_name,
    DEV_FOLDER, DEV_LABELS,
    lang_prefix='po_', lang_raw_ids=po_articles, gold_labels=po_gold)

print(f"\nEN: {len(en_true)} 段落")
print(f"PO: {len(po_true)} 段落")

In [ ]:
# ===== Step 5: 计算并汇报 Macro/Micro/Binary F1 =====
def report_metrics(y_true, y_pred, labels_name, lang_label):
    macro_f1  = f1_score(y_true, y_pred, average='macro',  zero_division=0)
    micro_f1  = f1_score(y_true, y_pred, average='micro',  zero_division=0)

    # Binary F1: 任意技术存在=positive
    bin_true = (y_true.sum(axis=1) > 0).astype(int)
    bin_pred = (y_pred.sum(axis=1) > 0).astype(int)
    from sklearn.metrics import f1_score as sk_f1
    binary_f1 = sk_f1(bin_true, bin_pred, average='binary', zero_division=0)

    print(f"\n{'='*55}")
    print(f"  {lang_label} 评估结果 (Sup-FT, volume-alignment (down-aligned) EN145/PL145)")
    print(f"{'='*55}")
    print(f"  Macro  F1 : {macro_f1*100:.2f}%")
    print(f"  Micro  F1 : {micro_f1*100:.2f}%")
    print(f"  Binary F1 : {binary_f1*100:.2f}%")
    print(f"{'='*55}")

    # per-technique
    per_tech = f1_score(y_true, y_pred, average=None, zero_division=0)
    tech_results = sorted(zip(labels_name, per_tech), key=lambda x: -x[1])
    print(f"  Per-technique F1:")
    for tech, f1 in tech_results:
        bar = '█' * int(f1 * 20)
        print(f"    {tech:<45} {f1:.3f}  {bar}")

    return {'macro_f1': macro_f1, 'micro_f1': micro_f1, 'binary_f1': binary_f1}

en_metrics = report_metrics(en_true, en_pred, labels_name, "English (EN145 down-aligned)")
po_metrics = report_metrics(po_true, po_pred, labels_name, "Polish  (PL145 original)")

# ===== 最终对比表 =====
print(f"\n{'='*65}")
print(f"  volume-alignment (down-aligned)实验 vs 论文Table 1 (Sup-FT)")
print(f"{'='*65}")
print(f"  {'设置':<30} {'Macro F1':>10} {'Micro F1':>10} {'Binary F1':>10}")
print(f"  {'-'*60}")
print(f"  {'EN(446) full  [paper]':<30} {'41.76%':>10} {'59.15%':>10} {'97.11%':>10}")
print(f"  {'EN(145) down-aligned':<30} {en_metrics['macro_f1']*100:>9.2f}% {en_metrics['micro_f1']*100:>9.2f}% {en_metrics['binary_f1']*100:>9.2f}%")
print(f"  {'-'*60}")
print(f"  {'PL(674) full  [paper]':<30} {'52.23%':>10} {'66.75%':>10} {'96.77%':>10}")
print(f"  {'PL(145) down-aligned':<30} {po_metrics['macro_f1']*100:>9.2f}% {po_metrics['micro_f1']*100:>9.2f}% {po_metrics['binary_f1']*100:>9.2f}%")
print(f"{'='*65}")
print()
print("解读:")
if en_metrics['macro_f1'] < 0.35:
    print("  EN(145) << EN(446) → 更多数据对EN确实有帮助")
    if po_metrics['macro_f1'] > en_metrics['macro_f1']:
        print("  EN(145) < PL(145) → 即使数据量相同PL仍更强 → 强化resource curse假说")
    else:
        print("  EN(145) ≥ PL(145) → 数据量是主要confound，resource curse假说被削弱")
else:
    print("  EN(145) ≈ EN(446) → 145篇已接近上限，FP集中来自annotation bias")

import json
eval_results = {
    'en_down145': en_metrics,
    'po_down145': po_metrics,
    'baseline': {
        'en536': {'macro_f1': 0.4176, 'micro_f1': 0.5915, 'binary_f1': 0.9711},
        'pl677': {'macro_f1': 0.5223, 'micro_f1': 0.6675, 'binary_f1': 0.9677},
    }
}
with open("your/path/here", "w") as f:
    json.dump(eval_results, f, indent=2)
print("\n✅ 结果已保存: your/path/here")